# Full-catalogue killer-whale call-type encode and site-controlled model (Colab GPU)

This notebook closes the encode gap for **Rung 1** of the decoding program: it
encodes **every** catalogue-labelled call-type detection in
`data/join_tables/call_type_manifest.csv` with the frozen **AVES2** encoder, then
runs the same site-controlled call-type classifiers we ran locally on the
1,954-segment join subset, but now over the **full 43-type catalogue**, including
the Northern-Resident (NRKW) N-calls that are not in the local embedding artifact.

**Pipeline.** For each provider it lists the GCS audio tree to build a
`basename -> object key` index (the manifest path omits the region subfolder),
then streams one source file at a time (download -> extract the labelled segments
-> encode -> delete), checkpointing to an `.npz`. It then trains:

- **Within-provider** call-type classifiers (recording site held constant):
  `vfpa` SRKW S-calls and `dfo_crp` NRKW N-calls, with a majority baseline and a
  label-permutation null.
- **Cross-provider transfer** for the shared SRKW S-calls (`vfpa` -> `smru`), the
  control the ecotype boundary failed.

**Scope.** This is call-type *discrimination* with catalogue labels that
correlate with pod/matriline; it is **not** evidence of meaning or of any
response to a broadcast signal. Runtime is dominated by audio download; use a GPU
runtime (Runtime -> Change runtime type -> GPU).

Data: DCLDE 2026 killer-whale corpus (NOAA/GCS public mirror). Catalogue codes are
Ford/Filatova stereotyped call types. References are keyed in `paper/refs.bib`
(`palmer2025dclde`, `hagiwara2023aves`, `ford1989`, `filatova2015`).

In [ ]:
#@title 1. Install dependencies, check GPU, and mount Drive (auto-persist)
# AVES2 encoder (avex) plus audio + ML stack. Colab already ships torch.
!pip -q install avex librosa soundfile gcsfs scikit-learn pandas matplotlib tqdm

import os, torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Set Runtime -> Change runtime type -> GPU for a fast encode.")

# Mount Google Drive so EVERY output (embedding checkpoint, metrics JSON, figure)
# is written straight to Drive and survives a session crash or idle timeout. The
# encode is resumable from the Drive checkpoint, so re-running just continues from
# where it stopped -- you never lose work and never need to be present to "save".
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTDIR = "/content/drive/MyDrive/OrcaDolittle_calltype"
except Exception as e:
    print("Drive mount unavailable, falling back to ephemeral /content:", e)
    OUTDIR = "/content/OrcaDolittle_calltype"
os.makedirs(OUTDIR, exist_ok=True)
print("All outputs will be saved to:", OUTDIR)

In [ ]:
#@title 2. Build the call-type manifest from public DCLDE annotations (no repo/auth needed)
# The repo is private, so instead of cloning we rebuild the manifest directly from
# the public DCLDE per-provider annotation CSVs on GCS (same logic as
# scripts/build_calltype_manifest.py). Fully anonymous.
import os, io, re, json, urllib.parse, urllib.request
from collections import Counter
import pandas as pd

os.makedirs("/content/work", exist_ok=True)
os.chdir("/content/work")

GCS_API = "https://storage.googleapis.com/storage/v1/b/noaa-passive-bioacoustic/o"
GCS_DL = "https://storage.googleapis.com/noaa-passive-bioacoustic/"
ROOT = "dclde/2027/dclde_2027_killer_whales"
PROVIDERS_WITH_CALLTYPE = ["dfo_crp", "smru", "vfpa"]
CALLTYPE_COLS = ["call_type", "Call Type", "calltype", "CallType"]
NON_CALL = {"", "unk", "unknown", "?", "n/a", "na", "none", "nothing", "nan"}

def list_csvs(provider):
    prefix = f"{ROOT}/{provider}/annotations/"
    out, token = [], None
    while True:
        url = f"{GCS_API}?prefix={urllib.parse.quote(prefix)}&maxResults=1000"
        if token:
            url += f"&pageToken={token}"
        d = json.load(urllib.request.urlopen(url, timeout=60))
        out += [it["name"] for it in d.get("items", []) if it["name"].endswith(".csv")]
        token = d.get("nextPageToken")
        if not token:
            break
    return out

def fetch_csv(name):
    raw = urllib.request.urlopen(GCS_DL + name, timeout=180).read()
    return pd.read_csv(io.BytesIO(raw), low_memory=False)

def normalise_call_type(val):
    s = str(val).strip()
    if s.lower() in NON_CALL:
        return None
    s = re.sub(r"\s*\(.*?\)\s*", "", s).replace(" call", "").replace("call", "").strip()
    s = s.rstrip("?").strip()
    if not s or s.lower() in NON_CALL:
        return None
    m = re.fullmatch(r"([A-Za-z]+)(\d+)([a-z]*)", s)
    if m:
        pref, num, suf = m.groups()
        s = f"{pref.upper()}{int(num):02d}{suf}"
    return s

def call_family(code):
    c = code.upper()
    return ("offshore" if c.startswith("OFF") else "NRKW" if c.startswith("N")
            else "SRKW" if c.startswith("S") else "Biggs/transient" if c.startswith("T")
            else "other")

rows = []
for pv in PROVIDERS_WITH_CALLTYPE:
    names = list_csvs(pv)
    print(f"{pv}: {len(names)} annotation CSVs")
    for name in names:
        df = fetch_csv(name)
        col = next((c for c in CALLTYPE_COLS if c in df.columns), None)
        if col is None:
            continue
        df = df[df[col].notna()].copy()
        if df.empty:
            continue
        df["call_type"] = df[col].map(normalise_call_type)
        df = df[df["call_type"].notna()]
        for _, r in df.iterrows():
            rows.append({
                "audio_path": r.get("path", r.get("filename", "")),
                "start": r.get("start", ""), "end": r.get("end", ""),
                "provider": pv, "kw_ecotype": r.get("kw_ecotype", ""),
                "clan": r.get("clan", ""), "subclan": r.get("subclan", ""),
                "pod": r.get("pod", ""), "call_type": r["call_type"],
                "call_family": call_family(r["call_type"]),
            })

man = pd.DataFrame(rows)
print(f"\nmanifest rows: {len(man)}")
print("by provider:", man["provider"].value_counts().to_dict())
print("by family:", man["call_family"].value_counts().to_dict())
print("distinct call types:", man["call_type"].nunique())

In [ ]:
#@title 3. Build basename -> GCS object-key index for each provider's audio
# The manifest stores an annotation-relative path that omits the deployment
# region subfolder (e.g. dfo_crp/audio/northbc/...), so we index every audio
# object by basename and match the manifest filename to it.
import json, urllib.parse, urllib.request

GCS_API = "https://storage.googleapis.com/storage/v1/b/noaa-passive-bioacoustic/o"
GCS_DL = "https://storage.googleapis.com/noaa-passive-bioacoustic/"
ROOT = "dclde/2027/dclde_2027_killer_whales"
AUDIO_EXT = (".wav", ".flac", ".aif", ".aiff")
PROVIDERS = sorted(man["provider"].unique().tolist())

def list_all_keys(prefix):
    keys, token = [], None
    while True:
        url = f"{GCS_API}?prefix={urllib.parse.quote(prefix)}&maxResults=1000"
        if token:
            url += f"&pageToken={token}"
        d = json.load(urllib.request.urlopen(url, timeout=120))
        for it in d.get("items", []):
            if it["name"].lower().endswith(AUDIO_EXT):
                keys.append(it["name"])
        token = d.get("nextPageToken")
        if not token:
            break
    return keys

basename_to_key = {}
for pv in PROVIDERS:
    keys = list_all_keys(f"{ROOT}/{pv}/audio/")
    for k in keys:
        basename_to_key[k.split("/")[-1]] = k
    print(f"{pv}: {len(keys)} audio objects indexed")

man["basename"] = man["audio_path"].astype(str).apply(lambda p: p.replace("\\", "/").split("/")[-1])
man["gcs_key"] = man["basename"].map(basename_to_key)
found = man["gcs_key"].notna()
print(f"\nmanifest detections with a resolved audio file: {found.sum()} / {len(man)}")
print("unresolved by provider:", man.loc[~found, "provider"].value_counts().to_dict())
man = man[found].copy()

In [ ]:
#@title 4. Load the frozen AVES2 encoder
from avex import load_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model("esp_aves2_sl_beats_all", return_features_only=True, device=DEVICE)
model.eval()
print("AVES2 encoder ready on", DEVICE)

In [ ]:
#@title 5. Stream, extract, and encode every labelled segment (resumable)
import time
import numpy as np
import librosa
from pathlib import Path
from tqdm.auto import tqdm

TARGET_SR = 16000
MIN_DUR_S = 1.0
CACHE = Path("/content/_audio_cache"); CACHE.mkdir(exist_ok=True)
# Checkpoint straight to Drive so a crash never loses encoded work.
OUT_NPZ = Path(OUTDIR) / "aves2_calltype_embeddings.npz"
CKPT_EVERY = 10

META_COLS = ["provider", "kw_ecotype", "clan", "subclan", "pod",
             "call_type", "call_family", "basename", "start", "end"]

def encode_waveform(wav):
    t = torch.from_numpy(wav).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        feats = model(t)
    return feats.cpu().numpy().mean(axis=1)[0]

# Resume from an existing checkpoint if present.
embeddings, metadata, done_keys = [], [], set()
if OUT_NPZ.exists():
    prev = np.load(OUT_NPZ, allow_pickle=True)
    embeddings = list(prev["embeddings"])
    metadata = list(prev["metadata"])
    done_keys = set(m.get("gcs_key", "") for m in metadata)
    print(f"resuming from {len(embeddings)} embeddings ({len(done_keys)} files done)")

groups = list(man.groupby("gcs_key"))
t0 = time.time()
processed = 0
for key, grp in tqdm(groups, desc="files"):
    if key in done_keys:
        continue
    local = CACHE / key.split("/")[-1]
    try:
        if not local.exists():
            urllib.request.urlretrieve(GCS_DL + key, str(local))
    except Exception as e:
        print("download fail", key.split("/")[-1], type(e).__name__)
        continue
    try:
        for _, r in grp.iterrows():
            start = float(r["start"]); end = float(r["end"])
            dur = max(end - start, MIN_DUR_S)
            try:
                wav, sr = librosa.load(str(local), sr=None, offset=max(start, 0.0),
                                       duration=dur, mono=True)
            except Exception:
                continue
            if sr != TARGET_SR:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
            need = int(MIN_DUR_S * TARGET_SR)
            if len(wav) < need:
                wav = np.pad(wav, (0, need - len(wav)))
            try:
                emb = encode_waveform(wav)
            except Exception:
                continue
            embeddings.append(emb)
            metadata.append({**{c: r[c] for c in META_COLS}, "gcs_key": key})
    finally:
        try:
            local.unlink()
        except Exception:
            pass
    processed += 1
    if processed % CKPT_EVERY == 0:
        np.savez_compressed(OUT_NPZ,
                            embeddings=np.array(embeddings, dtype=np.float32),
                            metadata=np.array(metadata, dtype=object))
        rate = len(embeddings) / max(time.time() - t0, 1e-6)
        print(f"  ckpt: {len(embeddings)} segs encoded, {processed} new files, {rate:.1f} segs/s")

np.savez_compressed(OUT_NPZ,
                    embeddings=np.array(embeddings, dtype=np.float32),
                    metadata=np.array(metadata, dtype=object))
X = np.array(embeddings, dtype=np.float32)
print(f"\nDONE: {X.shape} embeddings -> {OUT_NPZ}")
print("by family:", pd.Series([m["call_family"] for m in metadata]).value_counts().to_dict())
print("by provider:", pd.Series([m["provider"] for m in metadata]).value_counts().to_dict())

In [ ]:
#@title 6. Site-controlled call-type models on the full catalogue
import json
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 0
N_PERM = 200          # label-permutation null; raise for a tighter p-value
MIN_PER_TYPE = 30     # within-provider type inclusion threshold

meta = pd.DataFrame(metadata)
meta["row"] = np.arange(len(meta))

def clf():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=5000, class_weight="balanced"))

def mlp():
    return make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=600,
                                       alpha=1e-3, random_state=SEED))

def within_provider(provider, family, min_per_type=MIN_PER_TYPE):
    sub = meta[(meta["provider"] == provider) & (meta["call_family"] == family)]
    vc = sub["call_type"].value_counts()
    keep = vc[vc >= min_per_type].index
    sub = sub[sub["call_type"].isin(keep)]
    if sub["call_type"].nunique() < 2:
        return None
    Xs = X[sub["row"].to_numpy()]; y = sub["call_type"].to_numpy()
    classes = sorted(set(y)); k = len(classes)
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    yt, yp = [], []
    for tr, te in skf.split(Xs, y):
        yp.append(clf().fit(Xs[tr], y[tr]).predict(Xs[te])); yt.append(y[te])
    yt = np.concatenate(yt); yp = np.concatenate(yp)
    bal = balanced_accuracy_score(yt, yp); f1 = f1_score(yt, yp, average="macro")
    mt, mp = [], []
    for tr, te in skf.split(Xs, y):
        mp.append(mlp().fit(Xs[tr], y[tr]).predict(Xs[te])); mt.append(y[te])
    mlp_bal = balanced_accuracy_score(np.concatenate(mt), np.concatenate(mp))
    majority = pd.Series(y).value_counts(normalize=True).iloc[0]
    rng = np.random.default_rng(SEED); perm = []
    for it in range(N_PERM):
        ys = rng.permutation(y); a, b = [], []
        for tr, te in skf.split(Xs, ys):
            b.append(clf().fit(Xs[tr], ys[tr]).predict(Xs[te])); a.append(ys[te])
        perm.append(balanced_accuracy_score(np.concatenate(a), np.concatenate(b)))
        if (it + 1) % 50 == 0:
            print(f"    [{provider}/{family}] perm {it+1}/{N_PERM}")
    perm = np.array(perm); pval = (np.sum(perm >= bal) + 1) / (len(perm) + 1)
    return {"provider": provider, "family": family, "n": int(len(sub)), "k": k,
            "classes": classes, "balanced_accuracy": float(bal), "macro_f1": float(f1),
            "mlp_balanced_accuracy": float(mlp_bal), "majority_baseline": float(majority),
            "chance_1_over_k": 1.0 / k, "permutation_mean": float(perm.mean()),
            "permutation_pvalue": float(pval),
            "_cm": confusion_matrix(yt, yp, labels=classes)}

def cross_provider(train_pv, test_pv, family, min_train=30, min_test=10):
    s = meta[meta["call_family"] == family]
    tr = s[s["provider"] == train_pv]; te = s[s["provider"] == test_pv]
    tc = tr["call_type"].value_counts(); ec = te["call_type"].value_counts()
    shared = [t for t in tc.index if tc[t] >= min_train and ec.get(t, 0) >= min_test]
    if len(shared) < 2:
        return {"train": train_pv, "test": test_pv, "family": family,
                "shared_types": shared, "note": "insufficient shared types"}
    tr = tr[tr["call_type"].isin(shared)]; te = te[te["call_type"].isin(shared)]
    m = clf().fit(X[tr["row"].to_numpy()], tr["call_type"].to_numpy())
    yp = m.predict(X[te["row"].to_numpy()]); yt = te["call_type"].to_numpy()
    return {"train": train_pv, "test": test_pv, "family": family, "k": len(shared),
            "shared_types": shared, "n_train": int(len(tr)), "n_test": int(len(te)),
            "transfer_balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
            "transfer_macro_f1": float(f1_score(yt, yp, average="macro")),
            "chance_1_over_k": 1.0 / len(shared)}

results = {"within": {}, "transfer": {}}
for pv, fam in [("vfpa", "SRKW"), ("dfo_crp", "NRKW")]:
    print(f"== within {pv} {fam} ==")
    r = within_provider(pv, fam)
    if r:
        results["within"][f"{pv}_{fam}"] = r
        print(f"   n={r['n']} k={r['k']} linear={r['balanced_accuracy']:.3f} "
              f"mlp={r['mlp_balanced_accuracy']:.3f} chance={r['chance_1_over_k']:.3f} "
              f"maj={r['majority_baseline']:.3f} p={r['permutation_pvalue']:.2e}")
for tr_pv, te_pv, fam in [("vfpa", "smru", "SRKW")]:
    print(f"== transfer {tr_pv}->{te_pv} {fam} ==")
    r = cross_provider(tr_pv, te_pv, fam)
    results["transfer"][f"{tr_pv}_to_{te_pv}_{fam}"] = r
    if "transfer_balanced_accuracy" in r:
        print(f"   shared k={r['k']} transfer={r['transfer_balanced_accuracy']:.3f} "
              f"chance={r['chance_1_over_k']:.3f}")

In [ ]:
#@title 7. Figure and metrics JSON
from pathlib import Path
Path("figures").mkdir(exist_ok=True); Path("reports").mkdir(exist_ok=True)

within = results["within"]
n_cm = len(within)
fig, axes = plt.subplots(1, n_cm + 1, figsize=(7 * (n_cm + 1), 6))
if n_cm + 1 == 1:
    axes = [axes]
for ax, (name, r) in zip(axes[:n_cm], within.items()):
    cm = r["_cm"].astype(float)
    cmn = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    im = ax.imshow(cmn, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks(range(r["k"])); ax.set_yticks(range(r["k"]))
    ax.set_xticklabels(r["classes"], rotation=90, fontsize=6)
    ax.set_yticklabels(r["classes"], fontsize=6)
    ax.set_title(f"{r['provider']} {r['family']} ({r['k']} types)\n"
                 f"bal.acc={r['balanced_accuracy']:.2f} (chance {r['chance_1_over_k']:.2f}, "
                 f"p={r['permutation_pvalue']:.1e})", fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046)

labels, vals, chances = [], [], []
for name, r in within.items():
    labels.append(f"within\n{r['provider']} {r['family']}\n(k={r['k']})")
    vals.append(r["balanced_accuracy"]); chances.append(r["chance_1_over_k"])
for name, r in results["transfer"].items():
    if "transfer_balanced_accuracy" in r:
        labels.append(f"transfer\n{r['train']}->{r['test']}\n(k={r['k']})")
        vals.append(r["transfer_balanced_accuracy"]); chances.append(r["chance_1_over_k"])
xs = np.arange(len(labels))
axb = axes[-1]
axb.bar(xs, vals, width=0.5, color="#2a7fb8", label="balanced accuracy")
axb.plot(xs, chances, "r_", markersize=35, markeredgewidth=3, label="chance (1/k)")
axb.set_xticks(xs); axb.set_xticklabels(labels, fontsize=8); axb.set_ylim(0, 1)
axb.set_ylabel("balanced accuracy"); axb.set_title("Call-type discrimination vs chance")
for i, v in enumerate(vals):
    axb.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
axb.legend(fontsize=8)
plt.suptitle("Full-catalogue site-controlled killer-whale call-type classification (AVES2)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
FIG = "figures/calltype_model_full.png"
plt.savefig(FIG, dpi=150, bbox_inches="tight"); plt.show()

clean = {"within": {k: {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
                    for k, v in within.items()},
         "transfer": results["transfer"],
         "encoder": "AVES2", "n_segments": int(len(X)),
         "label_source": "DCLDE per-provider annotations (call_type)",
         "figure": FIG, "n_perm": N_PERM,
         "caveats": [
             "Within-provider designs hold recording site constant; transfer tests cross-site generalisation.",
             "Catalogue call types correlate with pod/matriline: call-type discrimination, NOT meaning.",
             "Encoded directly from public DCLDE audio; raw audio is not redistributed."]}
OUTJSON = "reports/calltype_model_full_summary.json"
Path(OUTJSON).write_text(json.dumps(clean, indent=2))

# Persist JSON + figure to Drive immediately (do not rely on an interactive download).
import shutil, os
for f in [FIG, OUTJSON]:
    shutil.copy(f, os.path.join(OUTDIR, os.path.basename(f)))
print("wrote", OUTJSON, "and", FIG)
print("auto-saved JSON + figure to Drive:", OUTDIR)
print(json.dumps(clean["within"], indent=2)[:1500])

In [ ]:
#@title 8. (Optional) download results -- they are ALREADY on your Drive
# Cells 5 and 7 saved the embedding .npz, the metrics JSON, and the figure to
# OUTDIR on your Google Drive, so nothing is lost even if the session ends or you
# step away. Bring these two files back into the repo (same paths) and commit:
#   reports/calltype_model_full_summary.json
#   figures/calltype_model_full.png
import os
print("Drive folder (persisted outputs):", OUTDIR)
for f in sorted(os.listdir(OUTDIR)):
    p = os.path.join(OUTDIR, f)
    print(f"   {f}  ({os.path.getsize(p)/1e6:.1f} MB)")

DOWNLOAD_NOW = True  #@param {type:"boolean"}
if DOWNLOAD_NOW:
    try:
        from google.colab import files
        files.download("reports/calltype_model_full_summary.json")
        files.download("figures/calltype_model_full.png")
    except Exception as e:
        print("Interactive download skipped:", e)
        print("Use the Drive copies in", OUTDIR, "instead.")